# Pidex — Pi Package Ecosystem Analysis

*Full dataset: 4,419 packages · npm keyword `pi-package` · June 2026*

In [ ]:
import json, re, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
import numpy as np
from IPython.display import display

sns.set_theme(style='whitegrid')
DATA  = Path('../data')
PLOTS = DATA / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

records = [json.loads(f.read_text()) for f in (DATA / 'enriched').glob('*.json')]
df = pd.DataFrame(records)
raw = json.loads((DATA / 'raw' / 'packages.json').read_text())
df['publisher'] = df['name'].map({p['name']: p.get('publisher') for p in raw})

def extract_slug(repo_field):
    url = repo_field.get('url','') if isinstance(repo_field, dict) else str(repo_field or '')
    m = re.search(r'github\.com[:/]([^/]+)/([^/.]+)', url)
    return f"{m.group(1)}__{m.group(2).rstrip('.git')}" if m else None

gh_rows = []
for _, row in pd.DataFrame(records).iterrows():
    slug = extract_slug(row.get('repository') or {})
    gh_file = DATA / 'github' / f'{slug}.json' if slug else None
    if gh_file and gh_file.exists():
        gh = json.loads(gh_file.read_text())
        gh_rows.append({'gh_stars': gh.get('stargazers_count'), 'gh_forks': gh.get('forks_count'),
                        'gh_open_issues': gh.get('open_issues_count'), 'gh_is_fork': gh.get('fork', False),
                        'gh_pushed_at': gh.get('pushed_at')})
    else:
        gh_rows.append({})

df = pd.concat([df, pd.DataFrame(gh_rows, index=df.index)], axis=1)
df['gh_pushed_at'] = pd.to_datetime(df['gh_pushed_at'], errors='coerce')

def infer_type(kws):
    for k in (kws or []):
        if k in ('pi-extension','pi-skill','pi-theme','pi-prompt'): return k.replace('pi-','')
    return 'unknown'
df['type'] = df['keywords'].apply(infer_type)

print(f"✓ Loaded {len(df)} packages  |  {df['gh_stars'].notna().sum()} with GitHub data")


## 1. Package Types

In [ ]:
type_counts = df['type'].value_counts()
fig, ax = plt.subplots(figsize=(7,4))
type_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title(f'Pi packages by type  (n={len(df)})'); ax.set_xlabel('')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()+10), ha='center')
display(type_counts.rename('count').to_frame())


## 2. Top Publishers

> `GitHub Actions` = 770 packages — automated CI publishes, many are forks/clones.

In [ ]:
pub_counts = df['publisher'].value_counts().dropna()
fig, ax = plt.subplots(figsize=(9,6))
pub_counts.head(20).sort_values().plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 20 publishers')
display(pub_counts.head(25).rename('packages').to_frame())


## 3. Download Distribution

Strong power law — the top 5 packages take ~50% of total downloads. Most packages get <100/week.

In [ ]:
dl = df[['name','downloads_last_week','type']].dropna(subset=['downloads_last_week']).sort_values('downloads_last_week', ascending=False)
print(f"{len(dl)} of {len(df)} packages have download data\n")
fig, axes = plt.subplots(1,2, figsize=(13,4))
dl['downloads_last_week'].plot(kind='hist', bins=40, ax=axes[0], title='Downloads/week (linear)', color='steelblue')
(dl['downloads_last_week']+1).plot(kind='hist', bins=40, ax=axes[1], title='Downloads/week (log scale)', logy=True, color='steelblue')
print("Top 25 by downloads/week:"); display(dl.head(25).reset_index(drop=True))


## 4. Download Trends & Trajectories

In [ ]:
trend_rows = [{'name': row['name'], 'week_end': e['week_end'], 'downloads': e['downloads']}
              for _, row in df.iterrows() for e in (row.get('download_trend') or [])]
tdf = pd.DataFrame(trend_rows)
tdf['week_end'] = pd.to_datetime(tdf['week_end'])
pivot = tdf.pivot_table(index='week_end', columns='name', values='downloads', aggfunc='sum').fillna(0)

fig, ax = plt.subplots(figsize=(13,4))
pivot.sum(axis=1).plot(ax=ax, color='steelblue')
ax.set_title('Total ecosystem downloads/week'); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

top_names = [n for n in dl.head(15)['name'].tolist() if n in pivot.columns]
fig, ax = plt.subplots(figsize=(13,6))
pivot[top_names].plot(ax=ax); ax.set_title('Weekly downloads — top 15 packages')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y')); ax.legend(loc='upper left', fontsize=7)


In [ ]:
def classify_trend(s):
    s = s[s > 0]
    if len(s) < 6: return 'new'
    r = s.iloc[len(s)//2:].mean() / s.iloc[:len(s)//2].mean()
    return 'growing' if r > 1.5 else ('declining' if r < 0.5 else 'stable')

trajectories = pivot.apply(classify_trend)
traj_counts = trajectories.value_counts()
colors_t = {'growing':'mediumseagreen','stable':'steelblue','new':'gold','declining':'tomato'}
fig, ax = plt.subplots(figsize=(6,3))
traj_counts.plot(kind='bar', ax=ax, color=[colors_t.get(i,'grey') for i in traj_counts.index])
ax.set_title('Package trajectory'); plt.tight_layout()
display(traj_counts.rename('packages').to_frame())


## 5. GitHub Stars

Stars and downloads diverge widely — both signals needed. Top stars leaders are big projects (Daytona, mem0, Context7) that added pi support.

In [ ]:
gh_has = df[df['gh_stars'].notna()].copy()
top_stars = gh_has.sort_values('gh_stars', ascending=False)
print(f"{len(gh_has)} packages with GitHub star data\n")
display(top_stars[['name','gh_stars','gh_forks','gh_is_fork','downloads_last_week','type']].head(25).reset_index(drop=True))


In [ ]:
scatter_df = gh_has.dropna(subset=['gh_stars','downloads_last_week'])
fig, ax = plt.subplots(figsize=(11,7))
ax.scatter(scatter_df['gh_stars'], scatter_df['downloads_last_week'], alpha=0.5, s=40, color='steelblue')
for _, r in scatter_df.nlargest(18,'gh_stars').iterrows():
    ax.annotate(r['name'].split('/')[-1], (r['gh_stars'], r['downloads_last_week']), fontsize=6.5, ha='left', va='bottom')
ax.set_xlabel('GitHub stars'); ax.set_ylabel('npm downloads/week')
ax.set_title('Stars vs Downloads  (log-log)'); ax.set_xscale('symlog'); ax.set_yscale('symlog')


In [ ]:
now = pd.Timestamp.now('UTC').tz_localize(None)
gh_has = gh_has.copy()
gh_has['days_since_push'] = (now - gh_has['gh_pushed_at'].dt.tz_localize(None)).dt.days
stale = gh_has[gh_has['days_since_push'] > 90].sort_values('downloads_last_week', ascending=False)
print(f"Stale packages (>90 days since last push): {len(stale)}")
display(stale[['name','days_since_push','downloads_last_week','gh_stars']].head(20).reset_index(drop=True))


## 6. Word Clouds

In [ ]:
def wc(texts, title, fname):
    combined = re.sub(r'[#*`\[\](){}|>]', ' ', ' '.join(t for t in texts if isinstance(t, str)))
    if not combined.strip(): return
    w = WordCloud(width=1200, height=400, background_color='white', max_words=120).generate(combined)
    plt.figure(figsize=(14,4)); plt.imshow(w, interpolation='bilinear')
    plt.axis('off'); plt.title(title); plt.tight_layout()

wc(df['description'].tolist(), 'All packages — descriptions', '07_wc_desc.png')
wc(df['readme'].fillna('').str[:800].tolist(), 'All packages — READMEs', '08_wc_readme.png')
for t in ['extension','skill','theme']:
    sub = df[df['type']==t]
    if len(sub) >= 3: wc(sub['description'].tolist(), f'{t} ({len(sub)} pkgs)', f'09_wc_{t}.png')


## 7. Near-Duplicate Detection (TF-IDF)

73,056 pairs above 0.25 cosine similarity. Top pairs are same-code published under two npm names.

In [ ]:
df['text'] = df['description'].fillna('') + ' ' + df['readme'].fillna('').str[:600]
vec = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))
tfidf = vec.fit_transform(df['text'])
sim = cosine_similarity(tfidf); np.fill_diagonal(sim, 0)

pairs = [{'pkg_a': df.iloc[i]['name'], 'pkg_b': df.iloc[j]['name'],
          'similarity': round(sim[i,j],3), 'dl_a': df.iloc[i]['downloads_last_week'],
          'dl_b': df.iloc[j]['downloads_last_week']}
         for i in range(len(df)) for j in range(i+1,len(df)) if sim[i,j] > 0.25]
pairs_df = pd.DataFrame(pairs).sort_values('similarity', ascending=False) if pairs else pd.DataFrame()
print(f"Near-duplicate pairs (similarity > 0.25): {len(pairs_df):,}")
display(pairs_df.head(30).reset_index(drop=True))


## 8. Cluster Analysis

In [ ]:
svd = TruncatedSVD(n_components=min(50,tfidf.shape[1]-1), random_state=42)
km = KMeans(n_clusters=12, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(svd.fit_transform(tfidf))
fn = vec.get_feature_names_out()
oc = km.cluster_centers_.argsort()[:,::-1]

rows = []
for i in range(12):
    cp = df[df['cluster']==i]; dls = cp['downloads_last_week'].dropna()
    total, mx = dls.sum(), (dls.max() if len(dls) else 0)
    ratio = mx/total if total > 0 else 0
    pat = 'blockbuster' if ratio>0.6 else ('contested' if ratio>0.35 else 'long-tail')
    rows.append({'C': i, 'pkgs': len(cp), 'pattern': pat,
                 'top_terms': ', '.join([fn[x] for x in oc[i,:4]]), 'total_dl': int(total)})

clusters_df = pd.DataFrame(rows)
display(clusters_df.sort_values('total_dl', ascending=False))

c_colors = {'blockbuster':'crimson','contested':'darkorange','long-tail':'steelblue'}
fig, ax = plt.subplots(figsize=(12,4))
clusters_df['label'] = clusters_df.apply(lambda r: f"C{r['C']}: {r['top_terms'].split(',')[0]}", axis=1)
clusters_df.set_index('label')['pkgs'].plot(kind='bar', ax=ax, color=[c_colors[p] for p in clusters_df['pattern']])
ax.set_title('Packages per cluster  (red=blockbuster · orange=contested · blue=long-tail)')


In [ ]:
for _, cr in clusters_df[clusters_df['pattern'].isin(['contested','blockbuster'])].sort_values('total_dl', ascending=False).iterrows():
    print(f"\n── Cluster {cr['C']} [{cr['pattern']}] — {cr['top_terms']}")
    cp = df[df['cluster']==cr['C']][['name','description','downloads_last_week','gh_stars','type']].sort_values('downloads_last_week', ascending=False)
    display(cp.head(8).reset_index(drop=True))


## 9. Fork Graph & Cross-References

In [ ]:
fork_graph = json.loads((DATA/'fork_graph.json').read_text())
cross_refs  = json.loads((DATA/'cross_refs.json').read_text())
print(f"Confirmed GitHub forks: {len(fork_graph)}")
display(pd.DataFrame(fork_graph).sort_values('stars', ascending=False).reset_index(drop=True))
print(f"\nPackages with README cross-references: {len(cross_refs)}")
top_refs = sorted(cross_refs, key=lambda x: x.get('downloads') or 0, reverse=True)
display(pd.DataFrame([{'package':c['package'],'downloads':c.get('downloads'),
                        'fork_signals':len(c.get('fork_signals',[])),
                        'pkg_mentions':len(c.get('mentions_packages',[]))} for c in top_refs[:25]]))


## 10. Summary

In [ ]:
print("=" * 55)
print("PHASE 0 — FINDINGS SUMMARY")
print("=" * 55)
print(f"  Total packages:          {len(df):,}")
print(f"  Type breakdown:          {df['type'].value_counts().to_dict()}")
print(f"  With GitHub data:        {df['gh_stars'].notna().sum():,}")
print(f"  Near-duplicate pairs:    {len(pairs_df):,}")
print(f"  Cluster patterns:        {clusters_df['pattern'].value_counts().to_dict()}")
print(f"  Trajectory breakdown:    {traj_counts.to_dict()}")
print(f"  Confirmed GitHub forks:  {len(fork_graph)}")
print(f"  README cross-refs:       {len(cross_refs)}")
print()
print("Top 5 by downloads/week:")
for _, r in dl.head(5).iterrows():
    print(f"  {r['name']:<45} {int(r['downloads_last_week']):>7,}/wk")
print()
print("Top 5 by GitHub stars:")
for _, r in top_stars.head(5).iterrows():
    print(f"  {r['name']:<45} {int(r['gh_stars']):>7,} ★")
